# Spectrum-SLM Training Analysis

This notebook provides interactive visualisations of the Spectrum-SLM training metrics. You can use it to track model accuracy, load model checkpoints, and run inference locally.

In [1]:
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from spectrum_slm_model import SpectrumSLM

In [2]:
# Path to the latest metrics JSON file saved during training
metrics_path = 'slm_checkpoints/metrics.json'

if os.path.exists(metrics_path):
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    
    print("--- Final Evaluation Metrics ---")
    print(f"PU Detection Accuracy: {metrics['pu_accuracy']*100:.2f}%")
    if not np.isnan(metrics['mod_accuracy']):
        print(f"Modulation Accuracy  : {metrics['mod_accuracy']*100:.2f}%")
    print(f"SNR MAE              : {metrics['snr_mae_db']:.2f} dB")
    
    # Plot per-SNR-bin accuracy for PU Detection
    snr_bins = list(metrics['per_snr_bin_pu_acc'].keys())
    acc = [metrics['per_snr_bin_pu_acc'][k] * 100 for k in snr_bins]
    
    plt.figure(figsize=(10, 5))
    plt.plot(snr_bins, acc, marker='o', linestyle='-', color='b')
    plt.title('PU Detection Accuracy vs SNR')
    plt.xlabel('SNR (dB)')
    plt.ylabel('Accuracy (%)')
    plt.grid(True)
    plt.show()
else:
    print("Metrics file not found. Let the training script finish running first.")

Metrics file not found. Let the training script finish running first.


## Quick Inference Test
Load the trained weights and simulate a single inference pass.

In [3]:
ckpt_path = 'slm_checkpoints/slm_phase2_best.pt'

model = SpectrumSLM(n_bins=176, patch_size=8, d_model=128, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1)

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
    model.eval()
    print("Trained model checkpoint loaded successfully!")
else:
    print("No checkpoint found natively. Generating random predictions...")
    model.eval()

# Mock input PSD (1 batch, 176 bins)
mock_input = torch.randn(1, 176)
with torch.no_grad():
    out = model(mock_input)

pu_prob = torch.softmax(out['pu_logits'], dim=1)[0, 1].item()
print(f"\nSample Input -> PU Detection Probability: {pu_prob*100:.1f}%")

Trained model checkpoint loaded successfully!

Sample Input -> PU Detection Probability: 100.0%


c:\Users\ASUS Vivo book\Desktop\Complete-Data-Science-With-Machine-Learning-And-NLP-2024-main\SDR_Data\spectrum_slm_model.py:153: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
